In [1]:
import random
import numpy as np
import gymnasium as gym

# Envirenment

In [2]:
env = gym.make(
    "FrozenLake-v1",
    is_slippery=False,
)

print("observation Space:", env.observation_space)
print("Action Space:", env.action_space)

observation Space: Discrete(16)
Action Space: Discrete(4)


In [3]:
NUM_STATES = env.observation_space.n
NUM_ACTIONS = env.action_space.n

ALPHA = 0.1
GAMMA = 0.9
EPSILON = 0.1

# Policy

In [4]:
def greedy_action(observation, Q):
    """
    Return the action with the larger estimated Q-value for specified state.
    """
    if np.max(Q[observation]) == np.min(Q[observation]):
        return random.randint(0, 3)
    else:
        return np.argmax(Q[observation])

In [5]:
def choose_action(observation, Q, env, epsilon=EPSILON):
    """
    Choose an action using epsilon-greedy exploration.
    """
    if random.random() < epsilon:
        return env.action_space.sample()
    else:
        return greedy_action(observation, Q)

# Q-Learning

In [6]:
def q_learning_update(
        Q, observation, action, reward, 
        next_observation, terminated, alpha=ALPHA, gamma=GAMMA
):
    """
    Q-Learning updating algorithm
    """
    if terminated:
        target = reward
    else:
        best_next_value = np.max(Q[next_observation])
        target = reward + gamma * best_next_value
    
    td_error = target - Q[observation, action]
    Q[observation, action] += alpha * td_error

    return Q, td_error

# Episode Generation

In [7]:
def run_training_episode(
        env, Q, 
        epsilon=EPSILON, alpha=ALPHA, gamma=GAMMA,
):
    observation , info = env.reset()
    total_reward = 0
    counter = 0
    while True:
        counter += 1
        action = choose_action(observation, Q, env, epsilon)
        next_observation, reward, terminated, truncated, info = env.step(action)
        Q, td_error = q_learning_update(Q, 
                                     observation, 
                                     action, 
                                     reward, 
                                     next_observation, 
                                     terminated,
                                     alpha,
                                     gamma,)
        total_reward += reward
        observation = next_observation
        done = terminated or truncated
        if done: 
            break
    return total_reward, Q, counter

In [8]:
num_episodes = 5000
episode_length = []
episode_rewards = []
Q = np.zeros((NUM_STATES, NUM_ACTIONS), dtype=float)

for episode in range(num_episodes):
    reward, Q, length = run_training_episode(env, Q)
    episode_length.append(length)
    episode_rewards.append(reward)

# Results

In [9]:
print("episode lengths:", episode_length)
print("Unique episode lengths:", np.unique(episode_length))
print("Mean episode length:", np.mean(episode_length))

episode lengths: [20, 13, 10, 10, 13, 5, 6, 10, 22, 19, 3, 27, 2, 4, 2, 6, 4, 3, 10, 4, 8, 14, 2, 2, 10, 9, 4, 8, 14, 2, 21, 6, 2, 13, 2, 5, 2, 25, 15, 7, 7, 22, 14, 7, 6, 8, 9, 8, 5, 20, 21, 5, 8, 19, 9, 12, 7, 5, 4, 10, 2, 5, 3, 5, 7, 6, 4, 10, 5, 4, 2, 15, 3, 2, 9, 5, 2, 3, 4, 3, 5, 2, 7, 16, 3, 12, 4, 9, 11, 7, 3, 17, 26, 17, 11, 11, 4, 4, 4, 6, 2, 3, 3, 13, 13, 5, 3, 8, 10, 14, 2, 6, 29, 9, 24, 14, 10, 3, 7, 28, 4, 2, 5, 8, 19, 3, 11, 6, 15, 6, 8, 2, 4, 27, 3, 2, 17, 2, 5, 2, 2, 8, 16, 3, 8, 8, 6, 6, 2, 24, 17, 10, 9, 3, 6, 13, 9, 4, 13, 5, 8, 4, 4, 16, 3, 5, 5, 3, 3, 2, 4, 8, 16, 3, 4, 8, 8, 3, 12, 4, 3, 9, 7, 3, 5, 14, 12, 15, 7, 18, 5, 3, 10, 9, 13, 14, 7, 15, 23, 9, 7, 6, 6, 8, 6, 6, 6, 7, 6, 13, 6, 5, 6, 6, 6, 10, 7, 6, 6, 4, 6, 6, 6, 3, 8, 7, 4, 6, 6, 8, 7, 6, 6, 6, 9, 6, 6, 6, 6, 6, 6, 6, 4, 6, 7, 6, 6, 6, 6, 7, 6, 18, 8, 8, 6, 6, 9, 6, 5, 6, 6, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 6, 6, 8, 6, 8, 7, 6, 6, 6, 6, 6, 7, 6, 6, 7, 9, 6, 10, 10, 7, 6, 6, 6, 6, 3, 6, 7, 6, 6, 6, 8, 6,

In [10]:
print("episode rewards:", episode_rewards)
print("Unique episode rewards:", np.unique(episode_rewards))
print("Mean episode rewards:", np.mean(episode_rewards))

episode rewards: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [11]:
print('Q:')
print(Q)

Q:
[[0.53144031 0.59049    0.4782962  0.53144089]
 [0.53144094 0.         0.00417678 0.12958306]
 [0.0887185  0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.59048764 0.6561     0.         0.53144071]
 [0.         0.         0.         0.        ]
 [0.         0.32334525 0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.65609996 0.         0.729      0.59048989]
 [0.65609861 0.81       0.80999697 0.        ]
 [0.44639864 0.9        0.         0.05607931]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.         0.80990607 0.9        0.72899621]
 [0.80999446 0.89999763 1.         0.8099969 ]
 [0.         0.         0.         0.        ]]


# Evaluation

In [27]:
def evaluate_policy(env, Q, num_episodes=100):

    episode_lengths = []
    episode_rewards = []
    for idx in range(num_episodes):
        observation , info = env.reset()
        total_reward = 0
        counter = 0

        while True:
            counter += 1
            action = greedy_action(observation, Q)
            next_observation, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            observation = next_observation
            done = terminated or truncated
            if done: 
                break

        episode_lengths.append(counter)
        episode_rewards.append(total_reward)

    return episode_rewards, episode_lengths

In [29]:
Q_before = Q.copy()
eval_total_reward, eval_length = evaluate_policy(env, Q)

assert np.allclose(Q, Q_before)

print('eval_total_reward')
print(eval_total_reward)
print('eval_length')
print(eval_length)

print("Success rate:", np.mean(eval_total_reward))
print("Mean evaluation length:", np.mean(eval_length))
print("Unique evaluation lengths:", np.unique(eval_length))

eval_total_reward
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
eval_length
[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
Success rate: 1.0
Mean evaluation length: 6.0
Unique evaluation lengths: [6]
